# A galactic-binary block for turntable

This notebook plugs a **real LISA source-class sampler** into the turntable
`Wheel`. To make it obvious *what is turntable and what is not*, everything that
is **not** turntable — the GBGPU waveform helpers, the LISA noise model, the data
injector, and the `GBBlock` itself — lives in a separate module,
[`gb_model.py`](gb_model.py). This notebook only:

1. imports turntable's `Residuals` and `Wheel`,
2. imports the model pieces from `gb_model`,
3. builds the dataset, wraps it in a `Residuals`, runs it through a `Wheel`, and
   reads the result back.

So every turntable touchpoint is visible here; the physics is behind the import.

- **waveforms** — [GBGPU](https://github.com/mikekatz04/GBGPU) (`master`), narrowband TDI;
- **sampler** — [Eryn](https://github.com/mikekatz04/Eryn), inside the block;
- **noise** — fixed LISA sensitivity, carried on `Residuals.noise`, consumed by
  the block via `residual.noise_psd()`.

> **Requires** the LISA stack (`gbgpu`, `eryn`, `lisaanalysistools`) plus
> `matplotlib` and `corner` for the plots, in the same env as `turntable` — the
> conda env, not turntable's core `.venv`. Outputs are not committed; run it to
> populate them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# turntable — the orchestration layer
from turntable import Residuals, Wheel

# everything that is NOT turntable (see gb_model.py)
from gb_model import FixedLISANoise, GBBlock, inject_gb, waveform

## 1. Define the dataset

turntable's frequency-domain convention is the one-sided rfft grid of the
underlying time series (`n_samples` counts *time-domain* samples; each tdi array
has length `n_samples // 2 + 1`).

In [ ]:
dt = 15.0
N_time = 2**19  # time-domain samples -> pins Tobs, df
Tobs = N_time * dt  # ~91 days
df = 1.0 / Tobs
freqs = np.fft.rfftfreq(N_time, dt)
CHANS = ("A", "E")
NB = 128  # GBGPU template band width (bins)

TRUTH = np.array([2.0e-22, 3.0e-3, 1.0e-16, 0.7])  # amp, f0, fdot, phi0
ANGLES = (0.9, 1.3, 2.1, 0.4)  # iota, psi, lam, beta  (fixed)
PARAMS = ["amp", "f0", "fdot", "phi0"]
print(f"Tobs={Tobs / 86400:.0f} d, df={df:.2e} Hz, grid={freqs.size} bins")

## 2. Noise model + injected data (both from `gb_model`)

`FixedLISANoise` and `inject_gb` are user code. `inject_gb` returns the raw
channel arrays; **wrapping them in a `Residuals` is where turntable enters** — and it
is where the fixed noise gets attached so the block can read it back.

In [ ]:
noise = FixedLISANoise()
data, info = inject_gb(TRUTH, ANGLES, Tobs, dt, N_time, CHANS, noise, NB=NB, seed=42)
print(
    f"injected GB at f0={TRUTH[1]} Hz, bins {info['band']}, optimal SNR={info['snr']:.1f}"
)

# --- turntable: the data contract ---
observed = Residuals(
    tdi=data,
    sample_rate=1.0 / dt,
    n_samples=N_time,
    channels=CHANS,
    tdi_generation="1.5",  # A1TDISens is the TDI 1.5 A channel
    observable="fractional_frequency",
    domain="frequency",
    noise=noise,
)  # epoch defaults to 0.0

## 3. Run the model through turntable

Register a `GBBlock` on a `Wheel` and run. The block reads channels, the
grid, and the noise (`residual.noise_psd()`) off the residual the Wheel hands
it — nothing is passed in from here except the residual itself.

In [ ]:
BOUNDS = [
    [0.3 * TRUTH[0], 3.0 * TRUTH[0]],
    [TRUTH[1] - 30 * df, TRUTH[1] + 30 * df],
    [0.0, 3.0e-16],
    [0.0, 2 * np.pi],
]

# MCMC sampling budget: total posterior samples = n_walkers * steps_per_cycle * n_cycles
n_walkers, steps_per_cycle, n_cycles = 40, 200, 12
block = GBBlock(
    init=TRUTH,
    bounds=BOUNDS,
    angles=ANGLES,
    band=NB,
    n_walkers=n_walkers,
    steps_per_cycle=steps_per_cycle,
)

wheel = Wheel(observed)  # turntable
wheel.add(block)  # turntable


def report(it, w):
    p = block.params
    print(
        f"  cycle {it + 1:2d}: amp={p[0]:.3e}  f0={p[1]:.9f}  fdot={p[2]:.3e}  phi0={p[3]:.3f}"
    )


total = n_walkers * steps_per_cycle * n_cycles
print(
    f"running the Wheel: {total:,} GB samples "
    f"({n_walkers} walkers x {steps_per_cycle} steps x {n_cycles} cycles), "
    f"noise via residual.noise_psd ..."
)
wheel.run(n_cycles, on_cycle=report)  # turntable drives block.block_update each cycle

## 4. Recovery

The posterior (half the chain after burn-in) sits within ~1σ of the injection.
`fdot` is intrinsically weakly constrained over 91 days. Cycle the sampling
budget up or down with `n_walkers`/`steps_per_cycle`/`n_cycles` above.

In [ ]:
post = block.chain[len(block.chain) // 2 :]  # drop first-half burn-in
print("posterior mean +/- std vs truth:")
for i, p in enumerate(PARAMS):
    m, s = post[:, i].mean(), post[:, i].std()
    z = (m - TRUTH[i]) / s if s else 0.0
    print(f"  {p:5s}: {m:.6e} +/- {s:.2e}   truth {TRUTH[i]:.6e}   ({z:+.1f} sigma)")

In [ ]:
import corner

fig = corner.corner(
    post,
    labels=PARAMS,
    truths=TRUTH,
    show_titles=True,
    title_fmt=".2e",
    label_kwargs={"fontsize": 9},
)
fig.suptitle("GB posterior (truth in blue)", y=1.02)
plt.show()

In [ ]:
band = info["band"]
rsi, rhA, rhE = waveform(post.mean(0), ANGLES, Tobs, dt, NB)
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(
    freqs[band] * 1e3, np.abs(observed.tdi["A"][band]), color="0.7", label="data |A|"
)
ax.semilogy(
    freqs[band] * 1e3,
    np.abs(info["signal"]["A"][band]),
    "C0",
    lw=2,
    label="injected |A|",
)
ax.semilogy(
    freqs[rsi : rsi + NB] * 1e3, np.abs(rhA), "C3--", lw=2, label="recovered |A|"
)
ax.set_xlabel("frequency [mHz]")
ax.set_ylabel("|A(f)|")
ax.legend()
ax.set_title("Galactic binary: data, injection, recovered model")
plt.tight_layout()
plt.show()

## Where next

- **All eight parameters** — enlarge `init`/`BOUNDS`/`ANGLES` (in `gb_model` the
  angles are held fixed) to sample the sky/orientation too.
- **A handful of sources** — register several `GBBlock`s on one `Wheel`, each with
  its own `name=` (the Wheel rejects duplicates), band, and Eryn sampler. Then the ledger earns its keep: the Wheel hands each GB the data
  minus every *other* GB's current model — the blocked-Gibbs
  global fit turntable orchestrates.
- **Sampled noise** — swap `FixedLISANoise` for a noise *block* that returns
  the residual with an updated `noise`; the GB block already re-reads
  `residual.noise_psd()` each `block_update`, so it tracks it for free.